# [LAB] Harness 보강 실습: 사람에게 묻고 승인받는 Deep Agent

`create_deep_agent`로 만든 Agent에 하네스 장치를 하나씩 더합니다. 모두 모델에게 부탁하는 대신 프레임워크가 지키게 하는 장치입니다.

- **2장 보내기 전에 승인받기**: 메일 도구를 부르기 직전에 Agent를 멈추고, 사람이 승인·수정·거절 중 하나를 고릅니다.
- **3장 Agent가 사람에게 묻기**: 질문 도구 `ask_user`로 Agent가 선택지를 내밀고, 사람이 고른 답으로 메일을 씁니다.
- **4장 터미널에서 메뉴로 답하기**: 2·3장을 합친 일정 조율 Agent를 파일로 만들고, 터미널에서 화살표 메뉴로 답합니다.
- **5장 고칠 수 있는 파일 정하기**: 파일 권한으로 Agent가 고칠 수 없는 폴더를 정합니다.
- **6장 조사는 하위 Agent에게 맡기기**: 검색은 하위 Agent가 하고, 메인 Agent는 정리된 답만 받습니다.

셀은 위에서부터 차례로 실행합니다. 장마다 첫머리의 접힌 준비 코드를 먼저 실행합니다. Agent를 실행하는 셀은 수십 초 걸리고, 답은 실행할 때마다 조금씩 다릅니다. 메일 도구는 연습용이라 실제로 메일을 보내지 않습니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. 실습 방법
- 2. 보내기 전에 승인받기
  - 2.1 Agent 만들기
  - 2.2 요청 보내고 멈춘 곳 보기
  - 2.3 결정 보내 이어 가기
- 3. Agent가 사람에게 묻기
  - 3.1 Agent 만들기
  - 3.2 요청 보내고 질문 받기
  - 3.3 답 골라 보내기
  - 3.4 승인하고 대화 보기
- 4. 터미널에서 메뉴로 답하기
  - 4.1 일정 조율 Agent 파일 쓰기
- 5. 고칠 수 있는 파일 정하기
  - 5.1 Agent 만들기
  - 5.2 규칙 파일 고쳐 달라고 하기
- 6. 조사는 하위 Agent에게 맡기기
  - 6.1 하위 Agent 정하기
  - 6.2 메인 Agent 만들고 맡기기
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 2026-10-21 추가. LangChain Academy `lca-deepagents`(python, 2026-10-19 커밋 1ea13fd) 모듈 1.8의 HITL 실습(`m1.8_hitl.py`, `m1.8_hitl_respond.py`)을 사내 상황(협력사 회의 메일)으로 옮기고, 같은 저장소의 파일 권한(`m2.2_agent.py`)·하위 Agent(`m4`, `m5`) 예제를 줄여 더했다. 원본 코드는 그대로 가져오지 않고 `dev/builders/labs/Chapter02/src/04-harness-hitl/`에 새로 썼다.
- 설치본 확인(deepagents 0.7.18, langchain 1.4.2, langgraph 1.2.12): `invoke(..., version="v2")`는 `GraphOutput(.value, .interrupts)`를 돌려준다. HITL 결정은 approve·edit·reject·respond 넷이고 허용하지 않은 결정을 보내면 `ValueError`. 기본 general-purpose 하위 Agent도 최상위 `interrupt_on`을 물려받는다(graph.py). `FilesystemPermission.mode`는 allow·deny·interrupt.
- questionary는 노트북 커널 안에서 멈춘다(2026-10-21 nbclient로 확인: 셀 시간 제한을 넘겨도 끝나지 않음). 그래서 2·3장은 결정을 값으로 주고, 화살표 메뉴는 4장 터미널 파일에서만 쓴다. `pyproject.toml`에 questionary를 넣었고, 노트북만 내려받은 수강생을 위해 명령은 `uv run --with questionary`로 쓴다. questionary가 없으면 번호 입력 메뉴로 동작한다.
- 사외 실측(2026-10-21, OpenRouter Qwen): 2장 약 15~25초, 3장 약 25초, 5장 약 10초, 6장 약 20초. 4장 터미널 파일은 pexpect로 끝까지 몰아 보았다: 질문 2번(조정 방식 → 빈 시간), 일정 조회 1번, 메일 거절(이유: 더 짧게) → 다시 쓴 메일 승인 → 끝.

</details>

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 보내기 전에 승인받기

메일 보내기처럼 되돌릴 수 없는 일은 Agent가 실행하기 전에 사람이 확인해야 합니다. `create_deep_agent`에 `interrupt_on`을 주면, Agent가 그 도구를 부르기 직전에 멈춥니다.

- **`interrupt_on`**: 멈출 도구 이름과, 사람이 고를 수 있는 결정(`allowed_decisions`)을 적습니다. `approve`는 그대로 실행, `edit`는 사람이 고친 값으로 실행, `reject`는 실행하지 않고 이유를 모델에게 전합니다.
- **체크포인터** `MemorySaver`: 멈춘 자리를 기억합니다. 이것이 있어야 사람의 결정을 받은 뒤 같은 대화(`thread_id`)를 이어서 실행할 수 있습니다.
- **`version="v2"`**: 실행 결과를 상태(`.value`)와 멈춘 곳(`.interrupts`)으로 나눠 받습니다. 멈췄으면 `.interrupts`에 검토할 도구 호출이 들어 있습니다.
- **`Command(resume=...)`**: 사람의 결정을 담아 멈춘 Agent를 이어서 실행합니다.

**왜 배우나:** 시스템 프롬프트에 "보내기 전에 물어봐"라고 적으면 모델이 지킬 때도 있고 안 지킬 때도 있습니다. `interrupt_on`은 프레임워크가 도구 실행 앞에서 반드시 멈추게 합니다.

준비 코드입니다. 필요한 도구를 불러오고, 연습용 메일 도구 `send_email`을 만듭니다. 이 도구는 실제로 메일을 보내지 않고, 보냈다면 무엇이 갔을지만 돌려줍니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
사람 승인 실습 1: 메일을 보내기 전에 사람에게 승인받기

create_deep_agent에 interrupt_on을 주면 Agent가 그 도구를 부르기 직전에 멈춘다.
사람이 approve(그대로)·edit(고쳐서)·reject(보내지 않음) 중 하나로 답하면 이어서 실행한다.
"""

import uuid

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from config.llm import get_llm

load_dotenv()


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """받는 사람에게 메일을 보낸다."""
    # 연습용 도구: 실제로 보내지 않고, 보냈다면 무엇이 갔을지만 돌려준다
    return f"메일을 보냈습니다. 받는 사람: {to} | 제목: {subject} | 본문: {body}"

### 2.1 Agent 만들기

In [ ]:
# ── Agent 만들기 ──────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """너는 DS부문 홍길동 프로의 메일을 대신 써서 보내는 업무 비서다.

규칙:
- 메일을 보내 달라는 요청에는 send_email을 쓴다.
- 메일은 홍길동 프로가 보내는 글로, 짧고 정중하게 쓰고 끝에 '홍길동 드림'을 붙인다.
- 도구 결과로 확인하기 전에는 메일을 보냈다고 말하지 않는다.
- 보냈다고 알릴 때는 원래 요청이 아니라 도구 결과에 있는 제목과 본문을 그대로 옮긴다.
"""

mail_agent = create_deep_agent(
    model=get_llm(),
    tools=[send_email],
    system_prompt=SYSTEM_PROMPT,
    # send_email을 부르기 직전에 멈추고, 사람의 결정(세 가지 중 하나)을 기다린다
    interrupt_on={"send_email": {"allowed_decisions": ["approve", "edit", "reject"]}},
    # 멈춘 자리를 기억해 두어야 사람의 결정을 받은 뒤 이어서 실행할 수 있다
    checkpointer=MemorySaver(),
)

### 2.2 요청 보내고 멈춘 곳 보기

In [ ]:
# ── 요청 보내고 멈춘 곳 보기 ──────────────────────────────────────────────────
# 실행할 때마다 새 대화(thread_id)로 시작한다
config = {"configurable": {"thread_id": f"mail-{uuid.uuid4().hex[:6]}"}}

# version="v2": 결과를 상태(.value)와 멈춘 곳(.interrupts)으로 나눠 받는다
mail_agent_result = mail_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "협력사 김민수 책임(minsu.kim@partner.example.com)에게 내일 오전 회의에 10분 늦는다고 메일 보내 줘.",
            }
        ]
    },
    config=config,
    version="v2",
)

if mail_agent_result.interrupts:
    pending = mail_agent_result.interrupts[0].value  # 사람이 검토할 도구 호출 목록
    request = pending["action_requests"][0]
    print("멈춘 도구:", request["name"])
    print("고를 수 있는 결정:", pending["review_configs"][0]["allowed_decisions"])
    for key, value in request["args"].items():
        print(f"- {key}: {value}")
else:
    print("멈추지 않고 끝났습니다. Agent가 send_email을 부르지 않았습니다.")
    print(mail_agent_result.value["messages"][-1].content)

### 2.3 결정 보내 이어 가기

**✍️ CHANGE_HERE: 결정을 edit(고쳐 보내기)나 reject(보내지 않기)로 바꾸면 메일은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `approve` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

사람이 고친 본문으로 보내기
```text
edit
```

보내지 않고 이유를 모델에게 전하기
```text
reject
```

값마다 위의 **요청 보내고 멈춘 곳 보기** 셀을 먼저 다시 실행해 새로 멈춘 뒤, 이 셀을 실행합니다. 한 번 이어 간 대화는 이미 끝났기 때문입니다. 다 봤으면 `approve`로 되돌립니다.

In [ ]:
# ── 결정 보내 이어 가기 ───────────────────────────────────────────────────────
# ✍️ CHANGE_HERE: 결정을 edit(고쳐 보내기)나 reject(보내지 않기)로 바꾸면 메일은 어떻게 될까요?
decision_type = "approve"

if not mail_agent_result.interrupts:
    print(
        "멈춘 곳이 없습니다. 위의 '요청 보내고 멈춘 곳 보기' 셀을 먼저 다시 실행합니다."
    )
else:
    if decision_type == "approve":  # 모델이 쓴 그대로 실행
        decision = {"type": "approve"}
    elif decision_type == "edit":  # 사람이 고친 값으로 실행
        edited_args = {
            **request["args"],
            "body": "김민수 책임님, 내일 오전 회의에 10분 늦게 들어가겠습니다. 죄송합니다. 홍길동 드림",
        }
        decision = {
            "type": "edit",
            "edited_action": {"name": request["name"], "args": edited_args},
        }
    else:  # 실행하지 않고, 이유를 모델에게 전한다
        decision = {
            "type": "reject",
            "message": "이 메일은 보내지 않습니다. 보내지 않았다고만 짧게 답하세요.",
        }

    mail_agent_result = mail_agent.invoke(
        Command(resume={"decisions": [decision]}), config=config, version="v2"
    )

    results = mail_agent_result.value["messages"]
    for m in results:
        m.pretty_print()
    if mail_agent_result.interrupts:
        print("\nAgent가 send_email을 다시 부르려고 또 멈췄습니다.")

<details><summary>무슨 일이 생기나</summary>

`edit`: `[Tool Message]`가 `Note: a human reviewer replaced this tool call before it ran. …`로 시작하고, 그 아래 `Tool response:`에 고친 본문(`김민수 책임님, 내일 오전 회의에 10분 늦게 들어가겠습니다. 죄송합니다. 홍길동 드림`)으로 보낸 결과가 나옵니다. 바로 위 `[Ai Message]`의 Tool Calls에는 모델이 처음 쓴 본문이 그대로 남아 있습니다. 실제로 실행된 것은 사람이 고친 값이고, 마지막 답도 고친 본문을 옮겼습니다(2026-10-21 사외 실행 2번).

`reject`: `[Tool Message]`가 `` User rejected the tool call for `send_email` with reason: 이 메일은 보내지 않습니다. … ``이고, 마지막 답은 `보내지 않았습니다.` 한 줄이었습니다(2번 모두). 메일 도구는 실행되지 않았습니다. 거절 이유는 모델에게 전하는 글이라, 이유에 따라 모델의 다음 행동이 달라집니다(4장에서 "더 짧게 다시 써 주세요"로 거절하면 다시 써 옵니다).

</details>

**볼 지점**
- 요청을 보낸 셀은 메일을 보내지 않고 멈췄습니다. `멈춘 도구: send_email`과 모델이 쓴 받는 사람·제목·본문이 출력됩니다. 아직 아무것도 실행되지 않았습니다.
- `approve`로 이어 가면 `[Tool Message]`에 `메일을 보냈습니다.`가 나오고, 마지막 `[Ai Message]`가 보낸 내용을 알려 줍니다.
- 결정은 도구마다 허용한 것만 쓸 수 있습니다. 이 Agent의 `send_email`에 `respond`를 보내면 오류가 납니다.
- 디버그 출력에 `[chain/error] … GraphInterrupt(…)` 줄이 보이면, Agent가 사람의 결정을 기다리며 멈춘 것입니다. 오류가 아니니 그대로 둡니다.

## 3. Agent가 사람에게 묻기

이번에는 도구를 하나 더합니다. 질문 도구 `ask_user`는 질문(`question`)과 선택지 목록(`choices`)을 받습니다. 이 도구에는 `respond`(사람이 답하기) 결정만 허용합니다.

- **도구는 실행되지 않습니다.** `respond`를 보내면 함수 몸통 대신 사람의 답이 도구 결과(`[Tool Message]`)로 들어갑니다. 그래서 몸통은 아무 글이나 돌려주는 자리표시입니다.
- **결정 네 가지는 한 목록에 적습니다.** approve·edit·reject·respond는 모두 `allowed_decisions`에 적고, 도구마다 말이 되는 것만 허용합니다. 질문 도구는 답하기만, 메일 도구는 승인·수정·거절만입니다.
- **선택지가 늘 있습니다.** `ask_user`는 부를 때마다 멈추고 선택지 목록을 함께 보내므로, 받는 쪽에서 버튼이나 메뉴로 보여 줄 수 있습니다. 4장에서 터미널 메뉴로 보여 줍니다.

요청은 "메일을 보내되, 회의를 다음 주로 미룰지 온라인으로 바꿀지 먼저 나한테 물어봐"입니다. Agent는 메일을 쓰기 전에 `ask_user`로 먼저 묻습니다.

**왜 배우나:** 시스템 프롬프트로 "먼저 물어봐"라고만 하면 모델이 묻지 않고 넘어가거나 자유 글로 물을 수 있습니다. 질문을 도구로 만들고 `respond`만 허용하면, 물을 때마다 반드시 멈추고 정해진 선택지 중 하나가 답으로 들어갑니다.

준비 코드입니다. 필요한 도구를 불러오고, 질문 도구 `ask_user`와 연습용 메일 도구 `send_email`을 만듭니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
사람 승인 실습 2: Agent가 사람에게 묻고 답을 받기

ask_user 도구는 respond 결정만 받게 해 둔다. 이 도구는 실제로 실행되지 않고,
사람이 고른 답이 도구 결과 자리에 그대로 들어간다.
"""

import uuid

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from config.llm import get_llm

load_dotenv()


@tool
def ask_user(question: str, choices: list[str]) -> str:
    """사용자에게 질문 하나와 고를 수 있는 선택지 목록을 보여 주고 하나를 고르게 한다."""
    # 이 줄은 실행되지 않는다. respond 결정이 사람의 답을 도구 결과로 대신 넣는다
    return f"[응답 없음: {question}]"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """받는 사람에게 메일을 보낸다."""
    # 연습용 도구: 실제로 보내지 않는다
    return f"메일을 보냈습니다. 받는 사람: {to} | 제목: {subject} | 본문: {body}"

### 3.1 Agent 만들기

In [ ]:
# ── Agent 만들기 ──────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """너는 DS부문 홍길동 프로의 메일을 대신 써서 보내는 업무 비서다.

규칙:
- 요청에 사용자만 정할 수 있는 선택이 있으면, 메일을 쓰기 전에 ask_user로 먼저 묻는다.
  question에는 질문 한 문장, choices에는 고를 수 있는 선택지를 2개 이상 넣는다.
- ask_user의 답에 맞춰 메일 내용을 정한다.
- 보낼 준비가 되면 send_email을 쓴다.
- 메일은 홍길동 프로가 보내는 글로, 짧고 정중하게 쓰고 끝에 '홍길동 드림'을 붙인다.
- 도구 결과로 확인하기 전에는 메일을 보냈다고 말하지 않는다.
"""

mail_agent = create_deep_agent(
    model=get_llm(),
    tools=[ask_user, send_email],
    system_prompt=SYSTEM_PROMPT,
    interrupt_on={
        # 묻는 도구: 사람이 답하는 것(respond)만 허용한다
        "ask_user": {"allowed_decisions": ["respond"]},
        # 밖으로 나가는 도구: 승인·수정·거절 중 하나
        "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
    },
    checkpointer=MemorySaver(),
)

### 3.2 요청 보내고 질문 받기

In [ ]:
# ── 요청 보내고 질문 받기 ─────────────────────────────────────────────────────
config = {"configurable": {"thread_id": f"ask-{uuid.uuid4().hex[:6]}"}}

mail_agent_result = mail_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "협력사 김민수 책임(minsu.kim@partner.example.com)에게 내가 내일 오후 회의에 참석하기 어렵다고 메일 보내 줘. "
                    "그런데 먼저 회의를 다음 주로 미뤄 달라고 할지, 온라인으로 바꿔 달라고 할지 나한테 물어봐."
                ),
            }
        ]
    },
    config=config,
    version="v2",
)

request = (
    mail_agent_result.interrupts[0].value["action_requests"][0]
    if mail_agent_result.interrupts
    else None
)
if request and request["name"] == "ask_user":
    print("질문:", request["args"]["question"])
    for i, choice in enumerate(request["args"]["choices"]):
        print(f"  [{i}] {choice}")
elif request:
    print(
        f"Agent가 묻지 않고 바로 {request['name']}을(를) 부르려 합니다. 이 셀을 한 번 더 실행합니다."
    )
else:
    print("멈추지 않고 끝났습니다. 이 셀을 한 번 더 실행합니다.")

### 3.3 답 골라 보내기

**✍️ CHANGE_HERE: 고르는 답을 두 번째 선택지로 바꾸면 메일 초안은 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `[0]` 자리에 복사해 붙여 넣기

```text
[1]
```

위의 **요청 보내고 질문 받기** 셀을 먼저 다시 실행해 새 질문을 받은 뒤, 이 셀을 실행합니다. `[0]`은 첫 번째, `[1]`은 두 번째 선택지입니다. 선택지 순서는 모델이 정하므로 실행마다 다를 수 있습니다. 다 봤으면 `[0]`으로 되돌립니다.

In [ ]:
# ── 답 골라 보내기 ────────────────────────────────────────────────────────────
# ✍️ CHANGE_HERE: 고르는 답을 두 번째 선택지로 바꾸면 메일 초안은 어떻게 달라질까요?
answer = request["args"]["choices"][0]

print("고른 답:", answer)
# respond: 도구를 실행하지 않고, 사람의 답을 도구 결과로 넣는다
mail_agent_result = mail_agent.invoke(
    Command(resume={"decisions": [{"type": "respond", "message": answer}]}),
    config=config,
    version="v2",
)

draft = (
    mail_agent_result.interrupts[0].value["action_requests"][0]
    if mail_agent_result.interrupts
    else None
)
if draft and draft["name"] == "send_email":
    print("\n보내기 전 메일 초안(승인을 기다리는 중):")
    for key, value in draft["args"].items():
        print(f"- {key}: {value}")
else:
    print(
        "메일 초안 단계에서 멈추지 않았습니다. '요청 보내고 질문 받기' 셀부터 다시 실행합니다."
    )

<details><summary>무슨 일이 생기나</summary>

이번 사외 실행에서 선택지는 `[0] 회의를 다음 주로 미뤄 달라고 요청`, `[1] 회의를 온라인으로 바꿔 달라고 요청`이었습니다. `[0]`의 초안은 "회의를 다음 주로 조정할 수 있도록 협조 부탁드릴 수 있을까 합니다", `[1]`의 초안은 "회의를 온라인으로 진행할 수 있도록 조정 부탁드려도 괜찮을지요"였습니다(2026-10-21). 두 경우 모두 `ask_user`의 `[Tool Message]`에는 고른 선택지 글이 그대로 들어 있었습니다. 모델은 사람의 답을 도구 결과로 읽고 메일 내용을 정합니다.

</details>

### 3.4 승인하고 대화 보기

In [ ]:
# ── 승인하고 대화 보기 ────────────────────────────────────────────────────────
mail_agent_result = mail_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}), config=config, version="v2"
)

results = mail_agent_result.value["messages"]
for m in results:
    m.pretty_print()

**볼 지점**
- 첫 셀은 메일을 쓰기 전에 멈추고 질문과 선택지를 보여 줍니다. 모델이 정한 질문이라 문장과 선택지 글은 실행마다 조금씩 다릅니다.
- 마지막 셀의 대화에서 `ask_user`의 `[Tool Message]`를 봅니다. 함수 몸통의 `[응답 없음: …]`이 아니라 고른 답이 그대로 들어 있습니다.
- 이 실습은 질문에 한 번, 메일에 한 번, 모두 두 번 멈춥니다. 몇 번 멈출지는 모델이 정하므로, 실제 앱에서는 멈출 때마다 결정을 받는 반복문을 씁니다(4장).

## 4. 터미널에서 메뉴로 답하기

2·3장을 합쳐 조금 더 큰 일정 조율 Agent를 만듭니다. 도구는 셋입니다: 빈 시간을 찾는 `check_calendar`(연습용 일정표), 질문 도구 `ask_user`, 메일 도구 `send_email`.

- Agent는 먼저 회의를 미룰지 온라인으로 바꿀지 묻습니다. 미루기를 고르면 빈 시간을 찾아 그 시간들을 선택지로 다시 묻습니다.
- 메일은 보내기 전에 보여 주고 승인·수정·거절을 고르게 합니다. 거절할 때 적은 이유는 Agent에게 전해져, Agent가 메일을 다시 써서 또 검토를 받습니다.
- 몇 번 멈출지 미리 알 수 없으므로 `while result.interrupts:` 반복문이 멈출 때마다 결정을 모아 이어서 실행합니다.
- 질문과 검토는 `questionary` 라이브러리의 화살표 메뉴로 보여 줍니다. questionary는 터미널 메뉴를 그리는 별도 라이브러리일 뿐이라, 웹 화면 버튼이나 메신저 버튼으로 바꿔도 Agent 쪽 코드는 그대로입니다.

화살표 메뉴는 노트북 셀 안에서는 동작하지 않습니다. 그래서 아래 셀로 파일을 만든 뒤 터미널에서 실행합니다.

**왜 배우나:** 사람 승인은 셀 하나로 끝나는 기능이 아니라 Agent 앱의 화면과 맞물리는 반복 구조입니다. 실제 업무용 Agent는 이 반복문 위에 화면을 얹습니다.

### 4.1 일정 조율 Agent 파일 쓰기

In [ ]:
%%writefile hitl_agent.py
"""
터미널에서 화살표 메뉴로 Agent에게 답하는 일정 조율 Agent

Agent가 묻는 질문(ask_user)은 화살표 메뉴로 고르고, 보내기 전 메일(send_email)은
승인·수정·거절 중에서 고른다. 거절할 때 적은 이유는 Agent에게 전해져 다시 쓰게 한다.
메뉴는 questionary 라이브러리로 그린다. 없으면 번호를 입력하는 메뉴로 대신한다.
"""

import uuid

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from config.llm import get_llm

load_dotenv()

try:
    import questionary
except ImportError:  # questionary가 없으면 번호 입력 메뉴로 동작한다
    questionary = None


# ── 도구 ──────────────────────────────────────────────────────────────────────
@tool
def check_calendar(week: str) -> str:
    """'이번 주' 또는 '다음 주'에 회의를 잡을 수 있는 빈 시간을 찾는다."""
    # 연습용 일정표: 실제 일정 시스템 대신 정해 둔 값을 돌려준다
    free = {
        "이번 주": ["금요일 16:00"],
        "다음 주": ["화요일 14:00", "수요일 10:00", "목요일 16:00"],
    }
    slots = free.get(week.strip(), free["다음 주"])
    return f"{week}에 비어 있는 시간: " + ", ".join(slots)


@tool
def ask_user(question: str, choices: list[str]) -> str:
    """사용자에게 질문 하나와 고를 수 있는 선택지 목록을 보여 주고 하나를 고르게 한다."""
    return f"[응답 없음: {question}]"  # 실행되지 않는다. 사람의 답이 이 자리에 들어간다


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """받는 사람에게 메일을 보낸다."""
    return f"메일을 보냈습니다(연습용이라 실제로 보내지 않음). 받는 사람: {to} | 제목: {subject} | 본문: {body}"


SYSTEM_PROMPT = """너는 DS부문 홍길동 프로의 회의 일정을 조율하고 메일을 보내는 업무 비서다.

규칙:
- 사용자만 정할 수 있는 선택은 ask_user로 묻는다. choices에는 선택지를 2~4개 넣는다.
- 회의를 미루기로 했다면 check_calendar로 빈 시간을 찾고, 그 시간들을 선택지로 ask_user에 다시 묻는다.
- 정해진 내용으로 메일을 써서 send_email로 보낸다. 메일은 홍길동 프로가 보내는 글로, 짧고 정중하게 쓰고 끝에 '홍길동 드림'을 붙인다.
- 메일이 거절되면 거절 이유를 반영해 다시 써서 send_email을 부른다. 이유에 보내지 말라고 했으면 보내지 않는다.
- 도구 결과로 확인하기 전에는 메일을 보냈다고 말하지 않는다.
"""

INTERRUPT_ON = {
    "ask_user": {"allowed_decisions": ["respond"]},
    "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
}

DEFAULT_REQUEST = (
    "협력사 김민수 책임(minsu.kim@partner.example.com)에게 내가 내일 오후 회의에 참석하기 어렵다고 메일 보내 줘. "
    "회의를 미룰지 온라인으로 바꿀지는 나한테 먼저 물어봐."
)


# ── 메뉴 ──────────────────────────────────────────────────────────────────────
def choose(question: str, choices: list[str]) -> str | None:
    """선택지 중 하나를 고르게 한다. Ctrl+C로 그만두면 None."""
    if questionary:
        return questionary.select(question, choices=choices).ask()
    print(question)
    for i, choice in enumerate(choices, start=1):
        print(f"  {i}. {choice}")
    while True:
        picked = input("번호: ").strip()
        if picked.isdigit() and 1 <= int(picked) <= len(choices):
            return choices[int(picked) - 1]
        print(f"1부터 {len(choices)} 사이의 번호를 입력합니다.")


def ask_text(question: str, default: str = "") -> str | None:
    """글을 입력받는다. 그냥 Enter를 누르면 default."""
    if questionary:
        return questionary.text(question, default=default).ask()
    typed = input(f"{question} (Enter: 그대로) ").strip()
    return typed or default


def review_email(request: dict) -> dict | None:
    """보내기 전 메일을 보여 주고 승인·수정·거절 중 하나를 고르게 한다."""
    args = request["args"]
    print("\n── 보내기 전 메일 ──")
    print(
        f"받는 사람: {args.get('to')}\n제목: {args.get('subject')}\n{args.get('body')}"
    )
    print("─" * 30)
    action = choose(
        "이 메일을 어떻게 할까요?",
        ["승인: 그대로 보내기", "수정: 본문 고쳐 보내기", "거절: 이유 적어 돌려보내기"],
    )
    if action is None:
        return None
    if action.startswith("승인"):
        return {"type": "approve"}
    if action.startswith("수정"):
        body = ask_text("고친 본문:", default=args.get("body", ""))
        if body is None:
            return None
        return {
            "type": "edit",
            "edited_action": {"name": request["name"], "args": {**args, "body": body}},
        }
    reason = ask_text(
        "거절 이유(Agent에게 전해집니다):", default="더 짧게 다시 써 주세요."
    )
    if reason is None:
        return None
    return {"type": "reject", "message": reason}


def print_new_steps(messages: list, start: int) -> int:
    """지난번 이후 새로 생긴 도구 호출과 도구 결과를 한 줄씩 보여 준다."""
    for m in messages[start:]:
        for call in getattr(m, "tool_calls", None) or []:
            print(f"  → Agent가 {call['name']} 호출: {call['args']}")
        if m.type == "tool":
            print(f"  ← {m.name} 결과: {str(m.content)[:120]}")
    return len(messages)


# ── 실행 ──────────────────────────────────────────────────────────────────────
def main() -> None:
    if questionary is None:
        print("(questionary가 없어 번호 입력 메뉴로 실행합니다)\n")
    agent = create_deep_agent(
        model=get_llm(),
        tools=[check_calendar, ask_user, send_email],
        system_prompt=SYSTEM_PROMPT,
        interrupt_on=INTERRUPT_ON,
        checkpointer=MemorySaver(),
    )
    config = {
        "configurable": {"thread_id": f"terminal-{uuid.uuid4().hex[:6]}"},
        "recursion_limit": 60,
    }

    user_request = ask_text("Agent에게 보낼 요청:", default=DEFAULT_REQUEST)
    if not user_request:
        return
    print("\nAgent가 일하는 중입니다(수십 초 걸릴 수 있습니다)...")
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_request}]},
        config=config,
        version="v2",
    )
    seen = print_new_steps(result.value["messages"], 0)

    # 멈출 때마다 사람의 결정을 모아 이어서 실행한다. 몇 번 멈추든 같은 반복문이 받는다
    while result.interrupts:
        decisions = []
        for request in result.interrupts[0].value["action_requests"]:
            if request["name"] == "ask_user":
                print()
                picked = choose(request["args"]["question"], request["args"]["choices"])
                decision = (
                    None if picked is None else {"type": "respond", "message": picked}
                )
            else:
                decision = review_email(request)
            if decision is None:
                print("\n중단했습니다. 메일은 보내지 않았습니다.")
                return
            decisions.append(decision)
        print("\nAgent가 이어서 일하는 중입니다...")
        result = agent.invoke(
            Command(resume={"decisions": decisions}), config=config, version="v2"
        )
        seen = print_new_steps(result.value["messages"], seen)

    print("\n── Agent의 마지막 답 ──")
    print(result.value["messages"][-1].content)


if __name__ == "__main__":
    main()

위 셀을 실행하면 이 폴더에 `hitl_agent.py` 파일이 만들어집니다. 터미널에서 실행합니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run --with questionary python course/Chapter02/notebooks/hitl_agent.py
```

3. `? Agent에게 보낼 요청:` 뒤에 정해 둔 요청이 채워져 있습니다. 그대로 Enter를 누릅니다.
4. 질문이 나오면 ↑↓ 키로 고르고 Enter를 누릅니다. 메일 검토 메뉴에서는 한 번은 **거절**을 골라 이유를 적고, 다시 써 온 메일을 승인해 봅니다.
5. 끝나면 `── Agent의 마지막 답 ──` 아래에 결과가 나옵니다. 중간에 그만두려면 **Ctrl+C**를 누릅니다.

`--with questionary`는 화살표 메뉴 라이브러리를 이번 실행에만 더해 씁니다. 설치 오류가 나면 `uv run python course/Chapter02/notebooks/hitl_agent.py`로 실행합니다. 그때는 번호를 입력하는 메뉴가 나옵니다.

**볼 지점**
- 터미널의 `→ Agent가 … 호출` 줄은 Agent가 부른 도구, `← … 결과` 줄은 도구 결과입니다. `ask_user` 결과에는 메뉴에서 고른 글이 그대로 들어갑니다.
- 거절 이유를 적으면 `← send_email 결과: User rejected the tool call … with reason: …`가 나오고, Agent가 이유에 맞춰 메일을 다시 써서 또 검토 메뉴를 띄웁니다.
- 질문 횟수와 문장은 모델이 정하므로 실행마다 다릅니다. 반복문은 몇 번 멈추든 같은 코드로 받습니다.

## 5. 고칠 수 있는 파일 정하기

Deep Agent는 작업 폴더의 파일을 읽고 쓰는 도구(`read_file`, `write_file`, `edit_file` 등)를 기본으로 가집니다. `permissions`에 `FilesystemPermission`을 적으면, 어떤 경로에 어떤 일(읽기 `read`·쓰기 `write`)을 할 수 있는지 정할 수 있습니다.

- **경로** `paths`: Agent에게 보이는 경로로 적습니다. 작업 폴더 `harness_plus`가 맨 위(`/`)로 보이므로, `/reference/**`는 `harness_plus/reference/` 아래 전부입니다.
- **`mode`**: `allow`(허용), `deny`(막음), `interrupt`(사람 승인을 기다림) 중 하나입니다.

아래 실습은 `/reference/`의 회의실 규칙 파일에 한 줄을 더해 달라고 합니다. 처음 값은 `/reference/` 쓰기를 막는 `deny`입니다.

**왜 배우나:** "규칙 파일은 고치지 마"를 프롬프트에 적는 것보다, 파일 도구가 그 경로에 쓰지 못하게 하는 편이 확실합니다. 막힌 도구는 오류를 돌려주고, 모델은 그 결과를 보고 답합니다.

준비 코드입니다. 필요한 도구를 불러오고, Agent의 작업 폴더 `harness_plus`와 그 아래 `reference/meeting_rules.md` 자리를 만듭니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
하네스 실습: 파일 권한으로 Agent가 고칠 수 있는 곳 정하기

FilesystemPermission은 Agent의 파일 도구(write_file·edit_file 등)가 어느 경로에 무엇을 할 수 있는지 정한다.
mode는 allow(허용)·deny(막음)·interrupt(사람 승인을 기다림) 셋 중 하나다.
"""

import uuid
from pathlib import Path

from deepagents import FilesystemPermission, create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

from config.llm import get_llm

load_dotenv()

WORK_DIR = Path(
    "harness_plus"
)  # Agent의 작업 폴더. Agent에게는 이 폴더가 맨 위(/)로 보인다
RULES_FILE = WORK_DIR / "reference" / "meeting_rules.md"
RULES_TEXT = """# 회의실 사용 규칙

- 회의실은 사내 예약 시스템으로 하루 전까지 예약한다.
- 회의가 끝나면 화이트보드를 지운다.
"""
RULES_FILE.parent.mkdir(parents=True, exist_ok=True)

### 5.1 Agent 만들기

**✍️ CHANGE_HERE: mode를 interrupt나 allow로 바꾸면 규칙 파일은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `deny` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

쓰기 전에 사람 승인을 기다리기
```text
interrupt
```

막지 않기
```text
allow
```

값마다 이 셀과 아래 **규칙 파일 고쳐 달라고 하기** 셀을 차례로 다시 실행합니다. 아래 셀은 실행할 때마다 규칙 파일을 처음 내용으로 되돌리고 시작합니다. 다 봤으면 `deny`로 되돌리고 이 셀을 다시 실행합니다.

In [ ]:
# ── Agent 만들기 ──────────────────────────────────────────────────────────────
file_agent = create_deep_agent(
    model=get_llm(),
    backend=FilesystemBackend(root_dir=str(WORK_DIR), virtual_mode=True),
    permissions=[
        FilesystemPermission(
            # ✍️ CHANGE_HERE: mode를 interrupt나 allow로 바꾸면 규칙 파일은 어떻게 될까요?
            operations=["write"],
            paths=["/reference/**"],
            mode="deny",
        ),
    ],
    system_prompt="너는 팀 문서를 정리하는 비서다. 파일을 고칠 수 없으면 그 이유를 짧게 말한다.",
    checkpointer=MemorySaver(),  # mode가 interrupt일 때 멈춘 자리를 기억한다
)

<details><summary>무슨 일이 생기나</summary>

`interrupt`: 모델이 `read_file`로 읽은 뒤 `write_file`을 부르자 그 자리에서 멈췄습니다. `사람 승인을 기다리며 멈췄습니다: write_file /reference/meeting_rules.md`가 나오고, 파일 내용은 그대로입니다. 이 셀은 결정을 보내지 않으므로 여기서 끝납니다. 2장처럼 `Command(resume=...)`로 approve를 보내야 그때 씁니다.

`allow`: `write_file`의 결과가 `Updated file /reference/meeting_rules.md`이고, 실행 뒤 파일 맨 아래에 `- 금요일 오후 회의실은 예약 없이 쓸 수 있다.`가 더해졌습니다. 다른 실행에서는 `write_file` 대신 `edit_file`로 고치기도 했습니다(2026-10-21 사외 실행).

</details>

### 5.2 규칙 파일 고쳐 달라고 하기

In [ ]:
# ── 규칙 파일 고쳐 달라고 하기 ────────────────────────────────────────────────
RULES_FILE.write_text(
    RULES_TEXT, encoding="utf-8"
)  # 실행할 때마다 처음 내용으로 되돌리고 시작한다
config = {"configurable": {"thread_id": f"perm-{uuid.uuid4().hex[:6]}"}}

file_agent_result = file_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "/reference/meeting_rules.md를 읽고, 맨 아래에 '- 금요일 오후 회의실은 예약 없이 쓸 수 있다.' 한 줄을 더해 줘.",
            }
        ]
    },
    config=config,
    version="v2",
)

results = file_agent_result.value["messages"]
for m in results:
    m.pretty_print()
if file_agent_result.interrupts:
    request = file_agent_result.interrupts[0].value["action_requests"][0]
    print(
        f"\n사람 승인을 기다리며 멈췄습니다: {request['name']} {request['args'].get('file_path')}"
    )

print("\n=== 실행 뒤 파일 내용 ===")
print(RULES_FILE.read_text(encoding="utf-8"))

**볼 지점**
- `deny`일 때 `write_file`(또는 `edit_file`)의 `[Tool Message]`가 `Error: permission denied for write on /reference/meeting_rules.md`입니다. 도구가 막혔고, 마지막 답은 고칠 수 없다고 말합니다.
- 맨 아래 `=== 실행 뒤 파일 내용 ===`은 실제 파일을 다시 읽은 것입니다. 모델이 뭐라고 답하든 파일이 바뀌었는지는 여기서 확인합니다.
- `read_file`은 막지 않았으므로 읽기는 됩니다. 막은 것은 `/reference/` 아래 쓰기뿐입니다.

## 6. 조사는 하위 Agent에게 맡기기

`subagents`에 하위 Agent를 적어 두면 메인 Agent가 `task` 도구로 일을 맡깁니다. 하위 Agent는 이름(`name`), 언제 맡길지 알려 주는 설명(`description`), 자기 시스템 프롬프트와 도구를 가집니다.

- 검색 도구 `search_news`는 하위 Agent `researcher`만 가집니다. 메인 Agent는 검색 도구가 없습니다.
- 하위 Agent는 따로 대화를 열어 검색 결과를 혼자 읽고, 정리한 답만 메인 Agent에게 돌려줍니다. 검색 원문은 메인 Agent의 대화에 들어오지 않습니다.
- 검색 도구는 연습용이라 정해 둔 가짜 기사 8건(약 1,500자)을 돌려줍니다.

**왜 배우나:** Agent의 대화가 길어질수록 모델이 읽어야 할 글이 늘어 느려지고 헷갈립니다. 긴 조사를 하위 Agent에게 맡기면 메인 Agent에는 필요한 결론만 남습니다.

준비 코드입니다. 필요한 도구를 불러오고, 연습용 검색 도구 `search_news`를 만듭니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
하네스 실습: 조사는 하위 Agent에게 맡기기

subagents에 하위 Agent를 적어 두면, 메인 Agent는 task 도구로 일을 맡긴다.
하위 Agent가 검색 결과를 혼자 읽고 정리한 답만 돌려주므로, 메인 Agent의 대화에는 긴 원문이 쌓이지 않는다.
"""

import uuid

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from config.llm import get_llm

load_dotenv()


@tool
def search_news(query: str) -> str:
    """반도체 업계 뉴스를 검색한다."""
    # 연습용 도구: 실제로 검색하지 않고, 정해 둔 가짜 기사 8건을 돌려준다
    articles = [
        "[기사 1] 메모리 업계, 고대역폭 메모리(HBM) 생산 능력 확대 계획을 잇달아 발표.",
        "[기사 2] AI 서버 수요 증가로 HBM 공급 부족이 내년까지 이어질 것이라는 전망.",
        "[기사 3] 차세대 HBM은 적층 단수를 늘려 용량을 키우는 방향으로 개발 중.",
        "[기사 4] 패키징 공정이 HBM 수율의 핵심 변수로 떠올라 관련 투자 확대.",
        "[기사 5] 고객사들이 HBM 공급처를 여러 곳으로 나누려는 움직임.",
        "[기사 6] 전력 효율이 HBM 경쟁의 새 기준으로 부각.",
        "[기사 7] 일부 고객사는 맞춤형 HBM 설계를 요구하기 시작.",
        "[기사 8] 장비 업계도 HBM용 본딩 장비 주문이 늘었다고 밝힘.",
    ]
    detail = (
        " 업계 관계자는 수요와 공급, 기술 일정, 고객사 요구를 함께 살펴야 한다고 말했다."
        * 3
    )
    return f"[검색어: {query}] (연습용 가짜 기사)\n" + "\n".join(
        a + detail for a in articles
    )

### 6.1 하위 Agent 정하기

In [ ]:
# ── 하위 Agent 정하기 ─────────────────────────────────────────────────────────
researcher = {
    "name": "researcher",
    "description": "주제 하나를 search_news로 검색해 정리해 돌려준다. 검색이 필요한 조사는 여기에 맡긴다.",
    "system_prompt": "너는 조사 담당이다. search_news로 검색한 뒤 핵심만 세 줄로 요약해 돌려준다. 검색 결과 원문은 돌려주지 않는다.",
    "tools": [search_news],  # 검색 도구는 하위 Agent만 가진다
}

### 6.2 메인 Agent 만들고 맡기기

In [ ]:
# ── 메인 Agent 만들고 맡기기 ──────────────────────────────────────────────────
main_agent = create_deep_agent(
    model=get_llm(),
    tools=[],  # 메인 Agent에게는 검색 도구를 주지 않는다
    subagents=[researcher],
    system_prompt="너는 팀장 보고를 돕는 비서다. 조사가 필요하면 task 도구로 알맞은 하위 Agent에게 맡기고, 받은 내용으로 답한다.",
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": f"sub-{uuid.uuid4().hex[:6]}"}}
main_agent_result = main_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "HBM 시장 동향을 조사해서 팀장님께 드릴 세 줄 요약을 만들어 줘.",
            }
        ]
    },
    config=config,
    version="v2",
)

results = main_agent_result.value["messages"]
for m in results:
    m.pretty_print()

# 메인 Agent가 누구에게 맡겼는지, 받은 답(task 도구 결과)이 얼마나 긴지 본다
calls = [
    c
    for m in results
    for c in getattr(m, "tool_calls", None) or []
    if c["name"] == "task"
]
task_replies = [m for m in results if m.type == "tool" and m.name == "task"]
print(
    "\n맡긴 하위 Agent:",
    [c["args"].get("subagent_type") for c in calls] or "없음(직접 답함)",
)
search_replies = [m for m in results if m.type == "tool" and m.name == "search_news"]
print(
    "메인 Agent 대화에 들어온 하위 Agent 답:",
    sum(len(str(m.content)) for m in task_replies),
    "자",
)
print(
    "메인 Agent 대화에 들어온 검색 원문:",
    sum(len(str(m.content)) for m in search_replies),
    "자",
)

**볼 지점**
- 메인 Agent의 대화에는 `task` 호출 하나와 그 결과(`[Tool Message]`, Name: task)만 있습니다. `task`의 `subagent_type`이 `researcher`이면 하위 Agent에게 맡긴 것입니다. 하위 Agent가 부른 `search_news`는 메인 대화에 보이지 않습니다.
- 맨 아래 줄을 비교합니다. 메인 Agent가 받은 하위 Agent의 답은 짧고(이번 실행에서 130~220자), 메인 대화에 들어온 검색 원문은 `0 자`입니다. 검색 원문 약 1,500자는 하위 Agent 안에서만 읽혔습니다.
- `task`의 `description`은 메인 Agent가 하위 Agent에게 쓴 지시입니다. 하위 Agent는 메인 대화를 보지 못하므로, 필요한 내용을 이 지시에 담아 보냅니다.

## 7. 정리

**이 노트북에서 한 것**
- `interrupt_on`으로 메일 도구를 부르기 직전에 Agent를 멈추고, 사람이 승인·수정·거절 중 하나로 이어 가게 했습니다.
- 질문 도구 `ask_user`에 `respond`만 허용해, Agent가 물을 때마다 반드시 멈추고 사람이 고른 선택지가 도구 결과로 들어가게 했습니다.
- 멈출 때마다 결정을 받는 반복문으로 일정 조율 Agent를 만들고, 터미널 화살표 메뉴로 답했습니다.
- 파일 권한(`FilesystemPermission`)으로 Agent가 고칠 수 없는 폴더를 정했습니다.
- 조사를 하위 Agent에게 맡겨 메인 Agent의 대화에 검색 원문이 쌓이지 않게 했습니다.
- 다섯 가지 모두 모델에게 부탁하는 문장이 아니라 하네스가 지키는 장치입니다. 프롬프트는 모델이 무엇을 할지 이끌고, 하네스는 무엇을 할 수 없는지 정합니다.